In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight

ana_xray = Path("../data/chest_xray/chest_xray")
ana_covid = Path("../data/covid19_radiography/COVID-19_Radiography_Dataset")

kayitlar = []
for bolum in ["train", "val", "test"]:
    for sinif in ["NORMAL", "PNEUMONIA"]:
        for yol in sorted((ana_xray / bolum / sinif).glob("*.jpeg")):
            kayitlar.append({"yol": yol, "bolum": bolum, "sinif": sinif})
df1 = pd.DataFrame(kayitlar)

kayitlar = []
for sinif in ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]:
    for yol in sorted((ana_covid / sinif / "images").glob("*.png")):
        kayitlar.append({"yol": yol, "sinif": sinif})
df2 = pd.DataFrame(kayitlar)
print("Veri 1 toplam:", len(df1), "| Veri 2 toplam:", len(df2))

Veri 1 toplam: 5856 | Veri 2 toplam: 21165


In [2]:
def alt_tur_bul(yol):
    ad = yol.name.lower()
    if "bacteria" in ad: return "BAKTERIYEL"
    if "virus" in ad: return "VIRAL"
    return "NORMAL"

def hasta_bul(yol):
    ad = yol.stem
    if ad.startswith("person"):
        return "_".join(ad.split("_")[:2])
    return ad.rsplit("-", 1)[0]

def md5(yol):
    return hashlib.md5(yol.read_bytes()).hexdigest()

df1["alt_tur"] = df1["yol"].apply(alt_tur_bul)
df1["hasta"] = df1["yol"].apply(hasta_bul)
df1["md5"] = df1["yol"].apply(md5)
df2["md5"] = df2["yol"].apply(md5)
print(df1["alt_tur"].value_counts().to_dict())
print("Hasta sayısı:", df1["hasta"].nunique())
print("Veri 1 kopya:", df1["md5"].duplicated().sum(), "| Veri 2 kopya:", df2["md5"].duplicated().sum())


{'BAKTERIYEL': 2780, 'NORMAL': 1583, 'VIRAL': 1493}
Hasta sayısı: 4168
Veri 1 kopya: 32 | Veri 2 kopya: 54


In [3]:
import re

def hasta_bul(yol):
    ad = yol.stem
    m = re.match(r"person(\d+)_(bacteria|virus)", ad)
    if m:
        return "P" + m.group(1) + "_" + m.group(2)
    m = re.search(r"IM-(\d+)", ad)
    if m:
        return "IM-" + m.group(1)
    return ad

df1["hasta"] = df1["yol"].apply(hasta_bul)
print("Hasta sayısı:", df1["hasta"].nunique())

Hasta sayısı: 3769


In [4]:
def on_isle(yol):
    img = Image.open(yol).convert("L")
    img = img.resize((224, 224))
    x = np.array(img, dtype=np.float32) / 255.0
    return np.stack([x, x, x], axis=-1)

ornek = df1["yol"].iloc[0]
ham = Image.open(ornek)
print("Önce :", ham.size, ham.mode, "| aralık:", np.array(ham).min(), "-", np.array(ham).max())
son = on_isle(ornek)
print("Sonra:", son.shape, son.dtype, "| aralık:", son.min(), "-", son.max())

Önce : (2090, 1858) L | aralık: 0 - 255
Sonra: (224, 224, 3) float32 | aralık: 0.0 - 0.9764706


In [5]:
egitim = df1[df1["bolum"] != "test"].reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
e_idx, d_idx = next(sgkf.split(egitim, egitim["alt_tur"], groups=egitim["hasta"]))
print("Veri 1:", len(e_idx), "eğitim /", len(d_idx), "doğrulama /", (df1["bolum"] == "test").sum(), "test")
print("Ortak hasta:", len(set(egitim.loc[e_idx, "hasta"]) & set(egitim.loc[d_idx, "hasta"])))

Veri 1: 4484 eğitim / 748 doğrulama / 624 test
Ortak hasta: 0


In [6]:
d2_egitim, d2_test = train_test_split(df2, test_size=0.15, stratify=df2["sinif"], random_state=42)
d2_egitim, d2_dogr = train_test_split(d2_egitim, test_size=0.15/0.85, stratify=d2_egitim["sinif"], random_state=42)
print("Veri 2:", len(d2_egitim), "/", len(d2_dogr), "/", len(d2_test))

siniflar = sorted(d2_egitim["sinif"].unique())
w = compute_class_weight("balanced", classes=np.array(siniflar), y=d2_egitim["sinif"])
print("Sınıf ağırlıkları:", dict(zip(siniflar, w.round(2))))

Veri 2: 14815 / 3175 / 3175
Sınıf ağırlıkları: {'COVID': np.float64(1.46), 'Lung_Opacity': np.float64(0.88), 'Normal': np.float64(0.52), 'Viral Pneumonia': np.float64(3.94)}
